# Лабораторная 3: YOLO-World (open-vocabulary detection)

Ноутбук включает примеры: базовый прогон YOLO-World с текстовыми классами, сравнение с YOLOv8, проверку устойчивости и заготовки для fine-tuning/GUI. Все шаги работают офлайн при наличии локальных весов и картинок.

## Запуск в Google Colab
1. Запустите ячейку установки зависимостей.
2. (Опционально) примонтируйте Google Drive и укажите путь к весам/датасетам.
3. Клонируйте/распакуйте репозиторий в `/content` или переименуйте путь `ROOT` при необходимости.

In [ ]:
# Установка зависимостей для Colab
!pip install -q ultralytics gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.2 MB/s eta 0:00:00


In [2]:
# (Опционально) монтирование Google Drive
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print('Drive смонтирован, укажите пути к весам/датасету в /content/drive/...')
except Exception as e:
    print('Colab drive недоступен или не нужен:', e)


Mounted at /content/drive
Drive смонтирован, укажите пути к весам/датасету в /content/drive/...


### Автозагрузка весов и демо-картинок (Colab)
Запустите ячейку ниже для скачивания минимально необходимых файлов в текущую сессию Colab. При локальном запуске можно пропустить.

In [ ]:
# Скачиваем веса и пару демо-картинок
import os
from pathlib import Path
import urllib.request

BASE_DIR = Path('/content/drive/MyDrive/Methods_for_processing_and_analyzing_heterogeneous_data')
if BASE_DIR.exists():
    ROOT = BASE_DIR / 'Lab3'
else:
    ROOT = Path.cwd() / 'Lab3'  # fallback for локального запуска
MODELS = ROOT / 'models'
DEMO = ROOT / 'data' / 'demo'
MODELS.mkdir(parents=True, exist_ok=True)
DEMO.mkdir(parents=True, exist_ok=True)

files = [
    ('https://huggingface.co/ultralytics/yolo-world/resolve/main/yolo_world_v2_m.pt', MODELS / 'yolov8m-worldv2.pt'),
    ('https://github.com/ultralytics/assets/releases/download/v0.0.0/yolov8n.pt', MODELS / 'yolov8n.pt'),
    ('https://ultralytics.com/images/bus.jpg', DEMO / 'bus.jpg'),
    ('https://ultralytics.com/images/zidane.jpg', DEMO / 'zidane.jpg'),
]

for url, path in files:
    if path.exists():
        print(f'Skip (exists): {path.name}')
        continue
    try:
        print('Downloading', url.split('/')[-1])
        urllib.request.urlretrieve(url, path)
    except Exception as e:
        print('Failed to download', url, '->', e)


Failed to download https://huggingface.co/ultralytics/yolo-world/resolve/main/yolo_world_v2_m.pt -> HTTP Error 401: Unauthorized
Skip (exists): yolov8n.pt


**Как запустить офлайн**
- Скачайте заранее веса `yolov8m-worldv2.pt` (и при желании `yolov8n.pt`) и положите в `Lab3/models/`.
- Подготовьте изображения: `Lab3/data/demo/` для свободного прогона, `Lab3/data/coco_subset/` (с аннотациями YOLO) для сравнения mAP/FPS, `Lab3/data/robust/` для тестов устойчивости.
- При отсутствии библиотек установите их вручную (например, `pip install ultralytics gradio` при наличии сети).

In [4]:
import os
import time
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFont

try:
    from ultralytics import YOLO, YOLOWorld
except Exception as e:
    YOLO = YOLOWorld = None
    print('Ultralytics не установлен или не загрузился:', e)

print('CUDA доступна:', torch.cuda.is_available())


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA доступна: True


In [5]:
# Пути и подготовка папок
BASE_DIR = Path('/content/drive/MyDrive/Methods_for_processing_and_analyzing_heterogeneous_data')
if BASE_DIR.exists():
    ROOT = BASE_DIR / 'Lab3'
else:
    ROOT = Path.cwd() / 'Lab3'  # fallback for локального запуска
DATA = ROOT / 'data'
MODELS = ROOT / 'models'
DEMO_DIR = DATA / 'demo'
COCO_DIR = DATA / 'coco_subset'
ROBUST_DIR = DATA / 'robust'

for d in (DATA, MODELS, DEMO_DIR, ROBUST_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Создаем простые плейсхолдеры, если в demo пусто (чтобы код не падал)
if not any(DEMO_DIR.iterdir()):
    for i, color in enumerate([(240, 80, 80), (80, 140, 240)]):
        img = Image.new('RGB', (640, 480), (245, 245, 245))
        draw = ImageDraw.Draw(img)
        draw.rectangle([120, 140, 520, 360], outline=color, width=8)
        draw.text((150, 150), f'placeholder {i+1}', fill=color)
        img.save(DEMO_DIR / f'placeholder_{i+1}.jpg')
    print('Созданы плейсхолдеры в', DEMO_DIR)


In [ ]:
# Функция загрузки YOLO-World
def load_yolo_world(weight_name='yolov8m-worldv2.pt'):
    if YOLOWorld is None:
        raise ImportError('Установите ultralytics >=8.2')
    local_path = MODELS / weight_name
    if not local_path.exists():
        raise FileNotFoundError(f'Положите веса {weight_name} в {MODELS}')
    model = YOLOWorld(local_path)
    return model

# Функция загрузки YOLOv8 для сравнения
def load_yolov8(weight_name='yolov8n.pt'):
    if YOLO is None:
        raise ImportError('Установите ultralytics >=8.2')
    local_path = MODELS / weight_name
    if not local_path.exists():
        raise FileNotFoundError(f'Положите веса {weight_name} в {MODELS}')
    return YOLO(local_path)


In [7]:
# Базовый прогон YOLO-World на пользовательских текстовых классах
custom_classes = ['electric scooter', 'coffee cup with logo', 'dog', 'person']  # включает классы вне COCO

try:
    yw_model = load_yolo_world()
    yw_model.set_classes(custom_classes)
    results = []
    for img_path in sorted(DEMO_DIR.glob('*.jpg')):
        start = time.time()
        res = yw_model.predict(img_path, imgsz=640, conf=0.25, verbose=False)[0]
        latency = time.time() - start
        res.speed['total_ms'] = latency * 1000
        results.append((img_path.name, res))
        res.save(filename=str(DEMO_DIR / f'pred_{img_path.name}'))
        print(f"{img_path.name}: {len(res.boxes)} боксов, {latency:.3f}s, сохранено -> pred_{img_path.name}")
except Exception as e:
    print('Пропускаю прогон YOLO-World (нет весов/ultralytics?):', e)
    results = []


Пропускаю прогон YOLO-World (нет весов/ultralytics?): Положите веса yolo_world_v2_m.pt в /content/drive/MyDrive/Methods_for_processing_and_analyzing_heterogeneous_data/Lab3/models


### Сравнение с YOLOv8 на 10 изображениях COCO
- Файл конфигурации `coco_subset.yaml` уже создан в `Lab3/data/coco_subset/`.
- Подготовьте датасет:
  - Положите минимум 10 картинок в `Lab3/data/coco_subset/images/train/`
  - Создайте соответствующие аннотации в формате YOLO в `Lab3/data/coco_subset/labels/train/`
  - Формат аннотации: `class_id x_center y_center width height` (все значения 0-1)
- Для быстрой подготовки минимального датасета запустите: `python data/coco_subset/prepare_minimal_dataset.py`
- Подробные инструкции см. в `data/coco_subset/README.md`
- Для mAP используем `model.val(data=...)` — Ultralytics сам посчитает mAP@0.5.
- FPS считаем вручную на тех же картинках.

In [ ]:
def evaluate_model(model, image_dir, cls_prompt=None):
    image_paths = sorted(list(Path(image_dir).glob('*.jpg')))
    if not image_paths:
        print('Нет изображений для оценки:', image_dir)
        return None
    timings = []
    for p in image_paths:
        start = time.time()
        kwargs = {'conf': 0.25, 'imgsz': 640, 'verbose': False}
        if cls_prompt is not None and hasattr(model, 'set_classes'):
            model.set_classes(cls_prompt)
        _ = model.predict(p, **kwargs)
        timings.append(time.time() - start)
    fps = len(timings) / sum(timings) if timings else 0
    return fps

try:
    coco_data_yaml = COCO_DIR / 'coco_subset.yaml'  # подготовьте yaml с путями
    if coco_data_yaml.exists():
        yw_map = None
        y8_map = None
        try:
            yw_model = load_yolo_world()
            yw_map = yw_model.val(data=coco_data_yaml, imgsz=640, plots=False, verbose=False).results_dict.get('metrics/mAP50(B)', None)
            yw_fps = evaluate_model(yw_model, COCO_DIR / 'images')
        except Exception as e:
            print('YOLO-World val пропущен:', e)
            yw_map = yw_fps = None
        try:
            y8_model = load_yolov8()
            y8_map = y8_model.val(data=coco_data_yaml, imgsz=640, plots=False, verbose=False).results_dict.get('metrics/mAP50(B)', None)
            y8_fps = evaluate_model(y8_model, COCO_DIR / 'images')
        except Exception as e:
            print('YOLOv8 val пропущен:', e)
            y8_map = y8_fps = None
        print('mAP@0.5: YOLO-World', yw_map, '| YOLOv8', y8_map)
        print('FPS: YOLO-World', yw_fps, '| YOLOv8', y8_fps)
    else:
        print('⚠️  Файл coco_subset.yaml не найден.')
        print(f'   Ожидается: {coco_data_yaml}')
        print('   Файл coco_subset.yaml уже создан в data/coco_subset/')
        print('   Для подготовки датасета см. data/coco_subset/README.md')
        print('   Или запустите: python data/coco_subset/prepare_minimal_dataset.py')
except Exception as e:
    print('Ошибка при сравнении моделей:', e)


Нет coco_subset.yaml — добавьте датасет для сравнения.


### Robustness: низкое разрешение, шум, ракурсы
Используем изображения в `Lab3/data/robust/`. Добавляем деградации и смотрим, как влияет текстовый запрос.

In [9]:
import random
import numpy as np

def degrade(img: Image.Image):
    # Понижаем разрешение, добавляем шум и поворот
    img_small = img.resize((img.width // 3, img.height // 3)).resize(img.size)
    arr = np.array(img_small).astype('float32')
    noise = np.random.normal(0, 15, size=arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype('uint8')
    noisy = Image.fromarray(arr).rotate(random.choice([-15, -7, 0, 7, 15]))
    return noisy

robust_images = sorted(list(ROBUST_DIR.glob('*.jpg')))
if not robust_images:
    print('Положите тестовые изображения в', ROBUST_DIR)

if robust_images and YOLOWorld is not None:
    try:
        yw_model = load_yolo_world()
        prompts = [['dog'], ['small brown dog']]
        for img_path in robust_images:
            img = Image.open(img_path).convert('RGB')
            degraded = degrade(img)
            degraded_path = img_path.with_name('degraded_' + img_path.name)
            degraded.save(degraded_path)
            for prm in prompts:
                yw_model.set_classes(prm)
                res = yw_model.predict(degraded_path, imgsz=640, conf=0.2, verbose=False)[0]
                res.save(filename=str(degraded_path.with_name(f"pred_{prm[0].replace(' ', '_')}_" + degraded_path.name)))
                print(f"{img_path.name} с промптом {prm}: {len(res.boxes)} боксов")
    except Exception as e:
        print('Robustness прогон пропущен:', e)


Положите тестовые изображения в /content/drive/MyDrive/Methods_for_processing_and_analyzing_heterogeneous_data/Lab3/data/robust


### Fine-tuning (дополнительно)
- Соберите минимум ~50 размеченных изображений, сформируйте `data.yaml` с путями.
- Рекомендуется взять базовые веса `yolov8m-worldv2.pt` и обучать 50–100 эпох с небольшим lr.

In [10]:
# Пример вызова (запускайте только когда готовы данные и есть ресурсы)
# try:
#     yw_model = load_yolo_world()
#     yw_model.train(data='path/to/data.yaml', epochs=50, imgsz=640, batch=8, lr0=0.0005, device=0)
# except Exception as e:
#     print('Fine-tune не запущен:', e)


### Gradio GUI (интерактивный ввод текста)
Если `gradio` установлен, можно запустить простой интерфейс. В офлайне работает при наличии весов и локальных изображений.

In [11]:
try:
    import gradio as gr
except Exception as e:
    gr = None
    print('Gradio не установлен:', e)

if gr and YOLOWorld is not None:
    yw_model = load_yolo_world()

    def detect(image, prompt: str):
        classes = [c.strip() for c in prompt.split(',') if c.strip()]
        if not classes:
            classes = ['person']
        yw_model.set_classes(classes)
        res = yw_model.predict(image, imgsz=640, conf=0.25, verbose=False)[0]
        plotted = res.plot()[:, :, ::-1]  # BGR->RGB
        return Image.fromarray(plotted)

    demo = gr.Interface(fn=detect,
                        inputs=[gr.Image(type='filepath'), gr.Textbox(label='Текстовые классы', value='person, dog')],
                        outputs=gr.Image(type='pil'),
                        title='YOLO-World demo')
    # demo.launch(share=False)
else:
    print('Gradio demo не запущен: нет gradio или ultralytics')


FileNotFoundError: Положите веса yolo_world_v2_m.pt в /content/drive/MyDrive/Methods_for_processing_and_analyzing_heterogeneous_data/Lab3/models